# pyMETRIC Batch Processing

UAV-based instantaneous ET estimation using the pyMETRIC energy balance model.

## Quickstart

1. **Drop your data** into a folder under `Batch_Data/`:
   ```
   Batch_Data/
     MyFlight_2025_06_15/
       config.txt              ← copy from config_template.txt and edit
       my_multiband_image.tif  ← UAV multispectral + thermal
       Input/
         endmember_zones.gpkg  ← optional: 'cold' and 'hot' polygons
         study_plots.gpkg      ← optional: zonal-stats polygons
   ```
2. **Edit `config.txt`** — Section 1 (per-flight) and Section 3 (crop preset) are the only blocks most users will need to modify.
3. **Run the cells below in order.** Each cell explains what it does.
4. **Look in `Output/`** when finished. This directory contains ET maps, reference ET, per-plot stats, and a processing log for each dataset.

## Cell map

| Cell | What it does | When to re-run |
|------|--------------|----------------|
| Setup | Imports, paths, deploys updated source | Once per session |
| Preflight | Lists datasets that are ready; sanity-checks each config | After adding/editing data |
| Run | Derives inputs, runs METRIC, writes outputs | Whenever you change a config or add a dataset |
| Summary | Per-dataset results table | After Run |
| Diagnostics | Input raster stats per dataset | When troubleshooting |
| Sensitivity | Re-runs each dataset with all 6 endmember strategies | Optional — slow |


In [ ]:
"""
Setup — imports and paths.

For a non-developer workflow: just open this notebook and Run All. The bundled
`pyMETRIC/` package that sits next to this notebook is loaded directly (placed
first on the import path), so the notebook always runs THIS folder's code. No
installation, no copying into site-packages, and any other `pyMETRIC` that may
be installed in the environment is ignored.

Edit BASE_DIR below ONLY if your data lives outside the notebook's directory.
"""
import os
import sys
import shutil

import numpy as np
import pandas as pd
from osgeo import gdal
gdal.UseExceptions()
from tqdm import tqdm

VERSION = '4.0.0'

# ---- Project paths --------------------------------------------------------
# Default: use the directory the notebook is in. Override only if needed.
BASE_DIR = os.getcwd()
# BASE_DIR = '/path/to/user_directory'  # ← uncomment + edit if data is elsewhere

# Find the folder that contains the bundled `pyMETRIC/` package.
if not os.path.exists(os.path.join(BASE_DIR, 'pyMETRIC')):
    if os.path.exists(os.path.join(os.path.dirname(BASE_DIR), 'pyMETRIC')):
        BASE_DIR = os.path.dirname(BASE_DIR)
    else:
        raise RuntimeError(
            'Could not find the bundled pyMETRIC/ package next to this '
            'notebook. Set BASE_DIR manually in this cell to the project '
            'folder that contains pyMETRIC/.')

BATCH_DIR = os.path.join(BASE_DIR, 'Batch_Data')
SOURCE_DIR = os.path.join(BASE_DIR, 'pyMETRIC')

# ---- Load the bundled source (always wins over any installed pyMETRIC) -----
# Put the project folder first on the import path and drop any cached pyMETRIC
# modules, so `import pyMETRIC` resolves to THIS folder's code. Nothing is
# written to disk and site-packages is left untouched.
sys.path.insert(0, BASE_DIR)
for _mod in [m for m in list(sys.modules)
             if m == 'pyMETRIC' or m.startswith('pyMETRIC.')]:
    del sys.modules[_mod]

import pyMETRIC
_loaded = os.path.dirname(os.path.abspath(pyMETRIC.__file__))
if _loaded != os.path.abspath(SOURCE_DIR):
    raise RuntimeError(
        'pyMETRIC was loaded from {}\n  but the bundled source is at {}.\n'
        '  The notebook is NOT running the bundled code. Restart the kernel '
        'and run this cell first.'.format(_loaded, SOURCE_DIR))

from pyMETRIC.METRICConfigFileInterface import METRICConfigFileInterface
from pyMETRIC.batch_helpers import (
    DatasetLogger, parse_config_simple, resolve_config_path,
    run_preflight_check, sanity_checks,
    derive_inputs_from_multiband,
    sanitize_lai, sanitize_temperature, sanitize_ndvi,
    validate_metric_output, convert_le_to_et, calc_etr_inst,
    compute_zonal_stats, check_image_stats, build_summary_table,
)

print(f'\npyMETRIC-batch v{VERSION}')
print(f'Using bundled pyMETRIC: {_loaded}')
print(f'Base directory:  {BASE_DIR}')
print(f'Batch directory: {BATCH_DIR} (exists: {os.path.isdir(BATCH_DIR)})')


In [2]:
"""
Preflight — scan Batch_Data/ for datasets and sanity-check each config.

Read-only. Won't modify any files. Use this to confirm what will be processed
before kicking off the Run cell.
"""
print('=== Preflight Check ===')
datasets = run_preflight_check(BATCH_DIR)

print('\n=== Config sanity check ===')
if not datasets:
    print('  (no datasets to check)')
for name, _, config_path in datasets:
    cfg = parse_config_simple(config_path)
    sanity_checks(name, cfg)


=== Preflight Check ===
  [Fruita_Example] READY — multiband: example_image_clipped.tif (config: config.txt)

1/1 datasets ready for processing.

=== Config sanity check ===
  Sanity check [Fruita_Example]:
  Acquisition: DOY 221, 10.50 hr -> Aug 9 at 10:30 (local standard time)


In [3]:
"""
Run — derive inputs, run METRIC, post-process. Per dataset:

  1. Derive NDVI / LAI / FC / TRAD from the multiband image (skipped if you
     already have pre-computed rasters in Input/).
  2. Sanitize the derived rasters.
  3. Run pyMETRIC energy balance model.
  4. Convert LE -> ET (mm/hr) for easier interpretation.
  5. Compute ASCE Penman-Monteith reference ET from the met values.
  6. Compute zonal stats if a polygon file is configured.
  7. Save a per-dataset processing log.

Per-dataset output goes into <dataset_folder>/Output/.
"""
batch_results = {}
et_results = []

for name, folder_path, config_path in tqdm(datasets, desc='Processing datasets'):
    cfg = parse_config_simple(config_path)
    output_dir = os.path.join(folder_path, 'Output')
    logger = DatasetLogger()

    print(f'\n{"="*60}\n  Dataset: {name}\n{"="*60}')

    # 1. Derive inputs from the multiband image
    with logger.capture('Derivation'):
        derive_ok = derive_inputs_from_multiband(folder_path, config_path)
    if not derive_ok:
        batch_results[name] = 'FAILED: derivation'
        logger.save(output_dir, name)
        continue

    # 2. Sanitize
    with logger.capture('Sanitization'):
        sanitize_lai(folder_path, lai_max=float(cfg.get('LAI_max', 8.0)))
        sanitize_temperature(folder_path)
        sanitize_ndvi(folder_path)

    # 3. Run pyMETRIC
    try:
        with logger.capture('METRIC Execution'):
            parser = METRICConfigFileInterface()
            cfg_data = parser.parse_input_config(config_path, is_image=True)
            parser.get_data(cfg_data, is_image=True)
            parser.run(is_image=True)

        status, msg = validate_metric_output(folder_path, name)
        if status == 'OK':
            batch_results[name] = 'SUCCESS'
        else:
            batch_results[name] = f'SUCCESS ({status}: {msg})'
            print(f'  Output issue: {msg}')
    except Exception as e:
        batch_results[name] = f'FAILED: {e}'
        print(f'  METRIC failed: {e}')
        logger.save(output_dir, name)
        continue

    # 4. LE -> ET (mm/hr)
    with logger.capture('LE to ET Conversion'):
        result = convert_le_to_et(folder_path, name)
        if result is not None:
            et_results.append(result)

    # 5. Reference ET
    with logger.capture('Reference ET'):
        required_met = ['T_A1', 'u', 'ea', 'p', 'S_dn', 'z_u']
        missing = [k for k in required_met if k not in cfg or not cfg[k].strip()]
        if missing:
            print(f'  Skipped ref ET: missing {", ".join(missing)}')
        else:
            try:
                etr = calc_etr_inst(cfg)
                T_units = cfg.get('T_A1_units', 'K').strip().upper()
                T_K = float(cfg['T_A1']) + 273.15 if T_units == 'C' else float(cfg['T_A1'])
                etr_row = {
                    'Dataset': name,
                    'Reference': cfg.get('reference_type', 'tall').strip().lower(),
                    'ETr_mm_hr': round(etr, 4),
                    'T_air_K': round(T_K, 2),
                    'Wind_ms': float(cfg['u']),
                    'S_dn_Wm2': float(cfg['S_dn']),
                }
                etr_csv = os.path.join(output_dir, name + '_Penman_ET.csv')
                pd.DataFrame([etr_row]).to_csv(etr_csv, index=False)
                print(f'  ETr = {etr:.4f} mm/hr -> {etr_csv}')
            except Exception as e:
                print(f'  Reference ET error: {e}')

    # 6. Zonal stats
    plots_path = resolve_config_path(cfg, 'zonal_polygons', folder_path)
    if plots_path and os.path.exists(plots_path):
        with logger.capture('Zonal Statistics'):
            dataset_stats = []

            import glob as _glob
            et_files = _glob.glob(os.path.join(output_dir, '*_ET_mm_hour.tif'))
            if et_files:
                stats = compute_zonal_stats(et_files[0], plots_path, name,
                                            value_label='ET', value_unit='mm/hr')
                n_valid = sum(1 for r in stats if r['Count'] > 0)
                print(f'  ET: {n_valid}/{len(stats)} features with valid data')
                dataset_stats.extend(stats)

            anc_files = [f for f in _glob.glob(os.path.join(output_dir, '*_METRIC_ancillary.tif'))
                         if '_CIMEC_' not in os.path.basename(f)
                         and '_ESA_' not in os.path.basename(f)
                         and '_MinMax_' not in os.path.basename(f)]
            if anc_files:
                ds_anc = gdal.Open(anc_files[0], gdal.GA_ReadOnly)
                if ds_anc is not None and ds_anc.RasterCount >= 9:
                    arr = ds_anc.GetRasterBand(9).ReadAsArray().astype(np.float32)
                    geo = ds_anc.GetGeoTransform()
                    prj = ds_anc.GetProjection()
                    rows_a, cols_a = arr.shape
                    ds_anc = None

                    tmp_path = os.path.join(output_dir, '_tmp_fETr.tif')
                    drv = gdal.GetDriverByName('GTiff')
                    ds_tmp = drv.Create(tmp_path, cols_a, rows_a, 1, gdal.GDT_Float32)
                    ds_tmp.SetGeoTransform(geo)
                    ds_tmp.SetProjection(prj)
                    ds_tmp.GetRasterBand(1).SetNoDataValue(float('nan'))
                    ds_tmp.GetRasterBand(1).WriteArray(arr)
                    ds_tmp.FlushCache()
                    ds_tmp = None

                    stats = compute_zonal_stats(tmp_path, plots_path, name,
                                                value_label='fETr', value_unit='unitless')
                    n_valid = sum(1 for r in stats if r['Count'] > 0)
                    print(f'  fETr: {n_valid}/{len(stats)} features with valid data')
                    dataset_stats.extend(stats)
                    os.remove(tmp_path)

            if dataset_stats:
                zonal_csv = os.path.join(output_dir, name + '_Zonal_Stats.csv')
                pd.DataFrame(dataset_stats).to_csv(zonal_csv, index=False)
                print(f'  Zonal stats -> {zonal_csv}')
    else:
        if plots_path:
            print(f'  Zonal stats skipped: {plots_path} not found')
        else:
            print('  Zonal stats skipped: zonal_polygons not set in config')

    logger.save(output_dir, name)

print(f'\n{"="*60}\nBatch finished. Run the Summary cell to see results.\n{"="*60}')


Processing datasets:   0%|                                                    | 0/1 [00:00<?, ?it/s]


  Dataset: Fruita_Example
  Image: example_image_clipped.tif (7 bands, 785x648, 0.2500 m)
  Bands: Red=3, NIR=5, Thermal=6
  NDVI: min=0.032, max=0.938, mean=0.317 (475520 valid pixels)
Using zone field: "zone" from /Users/sethmason/Library/CloudStorage/GoogleDrive-seth@lotichydrological.com/Shared drives/Projects/CSU/METRIC/pyMETRIC-uav/Batch_Data/Fruita_Example/Input/endmember_zones.gpkg
  Cold zone: 460 pixels from 1 polygon(s)
  Hot zone: 288 pixels from 1 polygon(s)
  NDVI_soil calibrated from hot zone: 0.122 (median of 288 pixels, default was 0.150)
  NDVI_full calibrated from cold zone: 0.924 (median of 460 pixels, default was 0.900)
  LAI: min=0.00, max=7.68, mean=0.81 (NDVI_soil=0.1216874476595833, NDVI_full=0.9236479462953083, k=0.6)
  FC:  min=0.001, max=0.990, mean=0.184
  Thermal: converted C -> K
  TRAD: min=293.28 K, max=329.83 K, mean=309.67 K
  Wrote NDVI.tif, LAI_NDVI.tif, FC.tif, TRAD.tif to Input/
Crop preset "alfalfa" applied: 8 parameter(s) (NDVI_full, h_C, k_ext

Processing datasets: 100%|████████████████████████████████████████████| 1/1 [00:02<00:00,  2.30s/it]

Saved Files
  ET mean = 0.2307 mm/hr
  ETr = 0.5267 mm/hr -> /Users/sethmason/Library/CloudStorage/GoogleDrive-seth@lotichydrological.com/Shared drives/Projects/CSU/METRIC/pyMETRIC-uav/Batch_Data/Fruita_Example/Output/Fruita_Example_Penman_ET.csv
  ET: 24/24 features with valid data
  fETr: 24/24 features with valid data
  Zonal stats -> /Users/sethmason/Library/CloudStorage/GoogleDrive-seth@lotichydrological.com/Shared drives/Projects/CSU/METRIC/pyMETRIC-uav/Batch_Data/Fruita_Example/Output/Fruita_Example_Zonal_Stats.csv
  Log saved: /Users/sethmason/Library/CloudStorage/GoogleDrive-seth@lotichydrological.com/Shared drives/Projects/CSU/METRIC/pyMETRIC-uav/Batch_Data/Fruita_Example/Output/Fruita_Example_log.txt

Batch finished. Run the Summary cell to see results.


Processing datasets: 100%|██████████| 1/1 [00:01<00:00,  1.84s/it]

Iteration: 3, non-converged pixels: 475520, max L diff: 0.019702, total time: 0.147588, loop time: 0.055900
Iteration: 4, non-converged pixels: 63971, max L diff: 0.002409, total time: 0.199807, loop time: 0.052219
Finished processing!
Saved Files
  ET mean = 0.2295 mm/hr
  ETr = 0.5267 mm/hr -> /Users/sethmason/Library/CloudStorage/GoogleDrive-seth@lotichydrological.com/Shared drives/Projects/CSU/METRIC/pyMETRIC-uav/Batch_Data/Fruita_Example/Output/Fruita_Example_Penman_ET.csv
  ET: 24/24 features with valid data
  fETr: 24/24 features with valid data
  Zonal stats -> /Users/sethmason/Library/CloudStorage/GoogleDrive-seth@lotichydrological.com/Shared drives/Projects/CSU/METRIC/pyMETRIC-uav/Batch_Data/Fruita_Example/Output/Fruita_Example_Zonal_Stats.csv
  Log saved: /Users/sethmason/Library/CloudStorage/GoogleDrive-seth@lotichydrological.com/Shared drives/Projects/CSU/METRIC/pyMETRIC-uav/Batch_Data/Fruita_Example/Output/Fruita_Example_log.txt

Batch finished. Run the Summary cell to se

In [4]:
"""
Summary — per-dataset results table.

Shows: status, mean ET (mm/hr), reference ETr, ET fraction (ET / ETr), and
the number of plot polygons that had zonal stats computed.
"""
summary = build_summary_table(batch_results, et_results, datasets)
print('=== Batch Summary ===')
display_cols = ['Dataset', 'Status', 'Mean ET (mm/hr)',
                'Ref ETr (mm/hr)', 'ET / ETr', 'Plots']
print(summary[display_cols].to_string(index=False))


=== Batch Summary ===
       Dataset  Status  Mean ET (mm/hr)  Ref ETr (mm/hr)  ET / ETr  Plots
Fruita_Example SUCCESS           0.2307           0.5267     0.438     24


In [5]:
"""
Diagnostics — input raster stats per dataset.

Use this to investigate data-quality issues (e.g. TRAD has too many NaNs,
or NDVI is clipped). Read-only; doesn't modify any files.
"""
for name, folder_path, _ in datasets:
    check_image_stats(folder_path)



=== Fruita_Example ===
  Multiband source: example_image_clipped.tif
    Bands: 7, Size: 785x648, Resolution: 0.2500 m
  Surface Temperature (TRAD.tif):
    Shape: (648, 785), Resolution: 0.2500 m
    Min: 293.28, Max: 329.83, Mean: 309.67, Std: 6.67
    NaN/NoData pixels: 33160 (6.5%)
  NDVI (NDVI.tif):
    Shape: (648, 785), Resolution: 0.2500 m
    Min: 0.03, Max: 0.94, Mean: 0.32, Std: 0.28
    NaN/NoData pixels: 33160 (6.5%)
  LAI (LAI_NDVI.tif):
    Shape: (648, 785), Resolution: 0.2500 m
    Min: 0.00, Max: 7.68, Mean: 0.81, Std: 1.86
    NaN/NoData pixels: 33160 (6.5%)
  Fractional Cover (FC.tif):
    Shape: (648, 785), Resolution: 0.2500 m
    Min: 0.00, Max: 0.99, Mean: 0.18, Std: 0.32
    NaN/NoData pixels: 33160 (6.5%)


In [6]:
"""
Cell 5: Endmember Sensitivity Analysis

Runs METRIC with multiple endmember selection strategies on each dataset
and compares the resulting ET estimates. Requires that Cell 4 (main processing)
has already completed derivation and sanitization — this cell reuses the
existing Input/ rasters and only varies the endmember configuration.

Strategies tested:
  - Min/Max (endmember_search=2) with zone polygons
  - Min/Max (endmember_search=2) full image (auto)
  - CIMEC (endmember_search=0) with zone polygons
  - CIMEC (endmember_search=0) full image (auto)
  - ESA (endmember_search=1) with zone polygons
  - ESA (endmember_search=1) full image (auto)

Each run produces a separate output tagged by strategy name.
"""

SENSITIVITY_STRATEGIES = [
    {'label': 'MinMax_zone',  'endmember_search': 2, 'endmember_mode': 'zone'},
    {'label': 'MinMax_auto',  'endmember_search': 2, 'endmember_mode': 'auto'},
    {'label': 'CIMEC_zone',   'endmember_search': 0, 'endmember_mode': 'zone'},
    {'label': 'CIMEC_auto',   'endmember_search': 0, 'endmember_mode': 'auto'},
    {'label': 'ESA_zone',     'endmember_search': 1, 'endmember_mode': 'zone'},
    {'label': 'ESA_auto',     'endmember_search': 1, 'endmember_mode': 'auto'},
]


def run_sensitivity_single(folder_path, config_path, strategy, output_suffix):
    """Run METRIC with a modified endmember strategy and collect ET stats.

    Parameters
    ----------
    folder_path : str
    config_path : str
    strategy : dict
        Keys: endmember_search, endmember_mode
    output_suffix : str
        Appended to output filename to distinguish runs

    Returns
    -------
    result : dict or None
    """
    # Parse the base config
    config_parser = METRICConfigFileInterface()
    config_data = config_parser.parse_input_config(config_path, is_image=True)

    # Override endmember settings
    config_data['endmember_search'] = str(strategy['endmember_search'])
    config_data['endmember_mode'] = strategy['endmember_mode']

    # If mode is 'auto', clear zone settings so they don't get picked up
    if strategy['endmember_mode'] == 'auto':
        config_data.pop('endmember_zones', None)
        config_data.pop('cold_zone', None)
        config_data.pop('hot_zone', None)

    # Set a unique output file
    folder_name = os.path.basename(folder_path)
    output_dir = os.path.join(folder_path, 'Output')
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    out_file = os.path.join(output_dir,
                            f'{folder_name}_METRIC_{output_suffix}.tif')
    config_data['output_file'] = out_file

    config_parser.get_data(config_data, is_image=True)
    config_parser.run(is_image=True)

    # Read the LE band (band 3) and compute ET stats
    if not os.path.exists(out_file):
        return None

    ds = gdal.Open(out_file, gdal.GA_ReadOnly)
    if ds is None or ds.RasterCount < 3:
        return None
    le = ds.GetRasterBand(3).ReadAsArray().astype(np.float64)
    ds = None

    # Also read TRAD for LE->ET conversion
    trad_file = os.path.join(folder_path, 'Input', 'TRAD.tif')
    ds_trad = gdal.Open(trad_file, gdal.GA_ReadOnly)
    if ds_trad is None:
        return None
    trad = ds_trad.GetRasterBand(1).ReadAsArray().astype(np.float64)
    ds_trad = None

    if trad.shape != le.shape:
        from scipy.ndimage import zoom
        trad = zoom(trad, (le.shape[0]/trad.shape[0], le.shape[1]/trad.shape[1]), order=1)

    t_celsius = trad - 273.15
    lambda_j = (2.501 - 0.002361 * t_celsius) * 1e6
    et = (le * 3600.0) / lambda_j
    et[et < 0] = 0.0
    et[~np.isfinite(et)] = np.nan

    valid = et[np.isfinite(et)]
    if len(valid) == 0:
        return None

    # Read the ancillary file for dT coefficients
    anc_file = out_file.replace('.tif', '_ancillary.tif')
    dT_info = ''
    if os.path.exists(anc_file):
        pass  # dT coefficients are printed to stdout during METRIC run

    return {
        'Strategy': output_suffix,
        'Search': ['CIMEC', 'ESA', 'MinMax'][strategy['endmember_search']],
        'Mode': strategy['endmember_mode'],
        'ET_mean_mm_hr': round(float(np.nanmean(valid)), 4),
        'ET_median_mm_hr': round(float(np.nanmedian(valid)), 4),
        'ET_std_mm_hr': round(float(np.nanstd(valid)), 4),
        'ET_min_mm_hr': round(float(np.nanmin(valid)), 4),
        'ET_max_mm_hr': round(float(np.nanmax(valid)), 4),
        'Valid_pixels': len(valid),
        'Output_file': os.path.basename(out_file),
    }


print('=== Endmember Sensitivity Analysis ===\n')

for name, folder_path, config_path in tqdm(datasets, desc='Datasets'):
    print(f'\n{"="*60}')
    print(f'  Dataset: {name}')
    print(f'{"="*60}')

    # Verify Input/ rasters exist (derivation must have run)
    input_dir = os.path.join(folder_path, 'Input')
    if not all(os.path.exists(os.path.join(input_dir, f))
               for f in ['TRAD.tif', 'NDVI.tif', 'LAI_NDVI.tif']):
        print(f'  Skipped: Input rasters not found. Run Cell 4 first.')
        continue

    output_dir = os.path.join(folder_path, 'Output')
    sensitivity_results = []

    for strategy in tqdm(SENSITIVITY_STRATEGIES,
                         desc=f'  {name} strategies', leave=False):
        label = strategy['label']
        print(f'\n  --- {label} ---')

        try:
            result = run_sensitivity_single(
                folder_path, config_path, strategy, label)
            if result is not None:
                result['Dataset'] = name
                sensitivity_results.append(result)
                print(f'    ET mean = {result["ET_mean_mm_hr"]:.4f} mm/hr')
            else:
                print(f'    No valid output produced')
        except Exception as e:
            print(f'    FAILED: {e}')

    # Save per-dataset sensitivity results
    if sensitivity_results:
        df = pd.DataFrame(sensitivity_results)
        csv_path = os.path.join(output_dir, f'{name}_sensitivity.csv')
        df.to_csv(csv_path, index=False)
        print(f'\n  Sensitivity results saved to {csv_path}')

        # Display comparison table
        display_cols = ['Strategy', 'Search', 'Mode',
                        'ET_mean_mm_hr', 'ET_median_mm_hr', 'ET_std_mm_hr']
        print(f'\n  {df[display_cols].to_string(index=False)}')

        # Summary statistics
        means = df['ET_mean_mm_hr']
        print(f'\n  ET mean range: {means.min():.4f} — {means.max():.4f} mm/hr '
              f'(spread: {means.max() - means.min():.4f})')
        print(f'  Coefficient of variation across strategies: '
              f'{means.std() / means.mean() * 100:.1f}%')
    else:
        print(f'\n  No valid results for sensitivity analysis')

print('\n=== Sensitivity Analysis Complete ===')


=== Endmember Sensitivity Analysis ===



Datasets:   0%|                                                               | 0/1 [00:00<?, ?it/s]


  Dataset: Fruita_Example



  Fruita_Example strategies:   0%|                                            | 0/6 [00:00<?, ?it/s]


  --- MinMax_zone ---
Crop preset "alfalfa" applied: 8 parameter(s) (NDVI_full, h_C, k_ext, landcover, rho_nir_C, rho_vis_C, tau_nir_C, tau_vis_C)
Auto-discovering input files in /Users/sethmason/Library/CloudStorage/GoogleDrive-seth@lotichydrological.com/Shared drives/Projects/CSU/METRIC/pyMETRIC-uav/Batch_Data/Fruita_Example/Input
  T_R1 -> TRAD.tif
  VI -> NDVI.tif
  LAI -> LAI_NDVI.tif
  f_c -> FC.tif
T_A1 converted: 27.38 C -> 300.53 K
Processing...
Estimating net shortwave radiation using Campbell two layers approach
CV window: 21 x 21 pixels (5.2 x 5.2 m at 0.250 m resolution)
Loading endmember zones from polygon file: /Users/sethmason/Library/CloudStorage/GoogleDrive-seth@lotichydrological.com/Shared drives/Projects/CSU/METRIC/pyMETRIC-uav/Batch_Data/Fruita_Example/Input/endmember_zones.gpkg
Endmember mode: zone
Reference surface: tall (ET0=alfalfa, G/Rn=0.040 for G_form=4)
Automatic search of METRIC hot and cold pixels
Cold pixel at (554, 376): 303.26 K, 0.929 VI (coldest 2.0

...

Cold pixel found at (539, 417) with 303.11 K and 0.933 VI
Hot pixel found at (44, 451) with 328.76 K and 0.079 VI
Endmember quality OK: LST range = 25.7 K, cold VI = 0.93, hot VI = 0.08
Allen 2013 safeguard: capping LE_cold from 406.15 to 357.71 (Rn-G=362.71, H_cold floored at 5.0 W m-2)
Calibrated dT: a=-49.349431, b=0.163366
Iteration: 0, non-converged pixels: 475520, max L diff: inf, total time: 0.000154, loop time: 0.000154
Iteration: 1, non-converged pixels: 475520, max L diff: 20.675053, total time: 0.047564, loop time: 0.047410
Iteration: 2, non-converged pixels: 475520, max L diff: 0.168517, total time: 0.101900, loop time: 0.054336
Iteration: 3, non-converged pixels: 475520, max L diff: 0.019617, total time: 0.161859, loop time: 0.059959


  Fruita_Example strategies: 100%|██████████| 6/6 [00:11<00:00,  1.90s/it]

Datasets: 100%|██████████| 1/1 [00:11<00:00, 11.12s/it]

Datasets: 100%|██████████| 1/1 [00:11<00:00, 11.12s/it]

Iteration: 4, non-converged pixels: 59418, max L diff: 0.002394, total time: 0.217336, loop time: 0.055477
Finished processing!
Saved Files
    ET mean = 0.1997 mm/hr

  Sensitivity results saved to /Users/sethmason/Library/CloudStorage/GoogleDrive-seth@lotichydrological.com/Shared drives/Projects/CSU/METRIC/pyMETRIC-uav/Batch_Data/Fruita_Example/Output/Fruita_Example_sensitivity.csv

     Strategy Search Mode  ET_mean_mm_hr  ET_median_mm_hr  ET_std_mm_hr
MinMax_zone MinMax zone         0.2295           0.2102        0.1238
MinMax_auto MinMax auto         0.2338           0.2149        0.1218
 CIMEC_zone  CIMEC zone         0.2222           0.2020        0.1280
 CIMEC_auto  CIMEC auto         0.1768           0.1464        0.1418
   ESA_auto    ESA auto         0.1997           0.1764        0.1341

  ET mean range: 0.1768 — 0.2338 mm/hr (spread: 0.0570)
  Coefficient of variation across strategies: 11.2%

=== Sensitivity Analysis Complete ===
